# ChestMedicalNet Training (Colab, Drive-backed)

Trains ChestMedicalNet: ConvNeXt-Base backbone + FPN + CBAM attention + disease-specific pathways (TB / Pneumonia / General) + Bayesian uncertainty + CORAL ordinal severity head, on NIH ChestX-ray14.

**This version keeps everything in Google Drive** (code + dataset + checkpoints), not the Colab VM's local disk, since free-tier Colab has limited temp space. Tradeoff: reading images from Drive during training is somewhat slower than local disk, but the ~45GB dataset only downloads once, ever, instead of every session.

**Read this before running:**
- NIH ChestX-ray14 has no TB label. The TB pathway exists architecturally but trains as a no-op (masked loss) on this dataset -- see `config/config.py::TB_LABEL_AVAILABLE`. Do not treat its output as a real TB classifier.
- ConvNeXt-Base is ~91M params total. Fits a free-tier T4 (16GB) with a modest batch size -- reduce `--batch-size` if you hit an out-of-memory error.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine, A100/L4 is faster). Run the next cell first and confirm `CUDA available: True`.

**One-time setup outside this notebook:**
1. Upload `chestmedicalnet_code.zip` (the project code, ~52KB) to your Google Drive at `MyDrive/ChestMedicalNet/chestmedicalnet_code.zip`. Re-upload it there (same path, overwrite) any time the code changes.
2. A free Kaggle account + API token: kaggle.com -> Settings -> Create New Token (downloads `kaggle.json`).
3. In this Colab notebook, click the key icon (Secrets) in the left sidebar and add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from `kaggle.json`).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('torch', torch.__version__, '| torchvision', end=' ')
import torchvision; print(torchvision.__version__)

## 1. Mount Google Drive and set up project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ChestMedicalNet'
CODE_ZIP = f'{PROJECT_DIR}/chestmedicalnet_code.zip'
CODE_DIR = f'{PROJECT_DIR}/code'
DATASET_CACHE_DIR = f'{PROJECT_DIR}/kagglehub_cache'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
LOG_DIR = f'{PROJECT_DIR}/logs'

for d in [PROJECT_DIR, CODE_DIR, DATASET_CACHE_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['CHECKPOINT_DIR'] = CHECKPOINT_DIR
os.environ['LOG_DIR'] = LOG_DIR

assert os.path.exists(CODE_ZIP), (
    f"{CODE_ZIP} not found -- upload chestmedicalnet_code.zip to "
    f"MyDrive/ChestMedicalNet/ first (see the setup note above), then re-run this cell."
)
print('Found code zip:', CODE_ZIP)

## 2. Unpack the code and install dependencies

Unzips fresh every run (`-o` overwrites), so re-uploading an updated zip to the same Drive path picks up code changes automatically. The code runs directly from Drive -- unlike the dataset's per-batch image reads, importing a few small `.py` files has no meaningful performance cost from being on Drive.

In [ ]:
!unzip -oq "$CODE_ZIP" -d "$CODE_DIR"
%cd $CODE_DIR

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

!pip install -q -r requirements-colab.txt

## 3. Download the dataset into Drive (one-time; skipped automatically on later runs)

`KAGGLEHUB_CACHE` points kagglehub's cache at a Drive folder instead of the Colab VM's local disk, so the ~45GB download happens once ever, not once per session. `DISABLE_COLAB_CACHE=1` turns off kagglehub's separate Colab-native mounting path (`/kaggle/input/...`), which is tied to Colab's own backend infrastructure and would NOT land in your Drive -- without this, kagglehub can silently ignore `KAGGLEHUB_CACHE` on a real Colab GPU runtime.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLEHUB_CACHE'] = DATASET_CACHE_DIR
os.environ['DISABLE_COLAB_CACHE'] = '1'

import glob
import kagglehub

raw_path = kagglehub.dataset_download("nih-chest-xrays/data")
print("kagglehub download path:", raw_path)
assert raw_path.startswith(DATASET_CACHE_DIR), (
    f"Expected the dataset under {DATASET_CACHE_DIR} but got {raw_path} -- "
    "KAGGLEHUB_CACHE was not respected, check the env vars above were set before this cell ran."
)

matches = glob.glob(os.path.join(raw_path, "**", "Data_Entry_2017.csv"), recursive=True)
assert matches, f"Data_Entry_2017.csv not found anywhere under {raw_path} -- download may have failed, check output above."
ARCHIVE_DIR = os.path.dirname(matches[0])
os.environ['ARCHIVE_DIR'] = ARCHIVE_DIR
print("Resolved ARCHIVE_DIR:", ARCHIVE_DIR)

In [ ]:
# Sanity-check the download landed in the expected layout.
!ls "$ARCHIVE_DIR" | head -20
!test -f "$ARCHIVE_DIR/Data_Entry_2017.csv" && echo 'Data_Entry_2017.csv found'
!ls "$ARCHIVE_DIR"/images_001/images | head -3
!du -sh "$DATASET_CACHE_DIR" 2>/dev/null

## 4. Pipeline sanity check (recommended before the full run)

A few epochs on a small subset -- confirms the full pipeline (data loading from Drive, model, loss, GPU training loop) works before committing GPU-hours to the full run.

In [ ]:
!python sanity_check.py --archive-dir "$ARCHIVE_DIR" --num-workers 2 --device cuda

## 5. Full training run

AdamW + cosine schedule with warmup, class-balanced focal loss, early stopping (patience 10) on validation mean per-class F1, up to 100 epochs. Checkpoints (`last_checkpoint.pt` every epoch, `best_model.pt` on improvement) go to Drive. Add `--batch-size N` to override the default if you hit an out-of-memory error.

In [ ]:
!python main.py train --model chest --architecture custom \
  --archive-dir "$ARCHIVE_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda

## 6. Resuming after a disconnected session

Everything (code, dataset cache, checkpoints) lives in Drive, so after a disconnect you only need to re-run cells 1-2 (mount + unzip code -- fast) and cell 3 (dataset download, which will find it already cached in Drive and skip re-downloading), then resume instead of restarting from epoch 0:

In [ ]:
!python main.py train --model chest --architecture custom \
  --archive-dir "$ARCHIVE_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --log-dir "$LOG_DIR" \
  --num-workers 2 \
  --device cuda \
  --resume "$CHECKPOINT_DIR/last_checkpoint.pt"